# Explore vibration spectra

Reads `backend/fft_backend.sqlite3` (written by `backend/analyze_fft.py`) and plots what's there. This is the exploratory step README §4 describes before any feature extraction, baseline comparison, or fault-detection logic gets written — look at real spectra first, by eye, against expected mechanical frequencies.

Nothing here writes back to the database.

In [ ]:
import json
import sqlite3
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

DB_PATH = Path("../backend/fft_backend.sqlite3")
conn = sqlite3.connect(DB_PATH)

In [ ]:
df = pd.read_sql_query(
    "SELECT id, window_id, device_id, analyzed_at, sample_rate_hz, "
    "freq_hz, fft_ax, fft_ay, fft_az, peak_axis, peak_freq_hz, peak_amp "
    "FROM fft_results ORDER BY analyzed_at",
    conn,
)
for axis in ("ax", "ay", "az"):
    df[f"fft_{axis}"] = df[f"fft_{axis}"].apply(json.loads)
df["freq_hz"] = df["freq_hz"].apply(json.loads)

print(f"{len(df)} analyzed window(s)")
df["device_id"].value_counts()

## Most recent spectrum per device

Sanity check: does the peak line up with an expected mechanical frequency (motor RPM, belt-pass frequency — README §2), or does it look like noise?

In [ ]:
devices = sorted(df["device_id"].unique())
fig, axes = plt.subplots(len(devices), 1, figsize=(9, 3 * len(devices)), squeeze=False)

for i, device_id in enumerate(devices):
    latest = df[df["device_id"] == device_id].iloc[-1]
    ax_plot = axes[i][0]
    ax_plot.plot(latest["freq_hz"], latest["fft_ax"], label="ax")
    ax_plot.plot(latest["freq_hz"], latest["fft_ay"], label="ay")
    ax_plot.plot(latest["freq_hz"], latest["fft_az"], label="az")
    ax_plot.set_title(f"{device_id} — peak {latest['peak_freq_hz']:.1f} Hz on {latest['peak_axis']}")
    ax_plot.set_xlabel("Hz")
    ax_plot.legend()

fig.tight_layout()

## Peak frequency over time, per device

Once there's enough history, this is where drift away from a healthy baseline would first show up — not meaningful yet with only a handful of windows, but the plot is here for when there is.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for device_id in devices:
    sub = df[df["device_id"] == device_id]
    ax.plot(pd.to_datetime(sub["analyzed_at"], unit="s"), sub["peak_freq_hz"], marker="o", label=device_id)
ax.set_xlabel("analyzed at")
ax.set_ylabel("peak frequency (Hz)")
ax.legend()
fig.tight_layout()